# ***03 Detection***

`EDA and Feature Engineering is Done!`

---
## ***1. Loading data***

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest

In [2]:
# Scaled Data

train_dataset = pd.read_csv("../data/scaled_train_data.csv")
test_dataset = pd.read_csv("../data/scaled_test_data.csv")

X_train = train_dataset.drop(columns=["Price","Unnamed: 0"])
y_train = train_dataset["Price"]

X_test = test_dataset.drop(columns=["Price","Unnamed: 0"])
y_test = test_dataset["Price"]

In [3]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112723 entries, 0 to 112722
Data columns (total 20 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   target__Brand               112723 non-null  float64
 1   target__Model Name          112723 non-null  float64
 2   target__Model Variant       112723 non-null  float64
 3   target__State               112723 non-null  float64
 4   onehot__Fuel Type_CNG       112723 non-null  float64
 5   onehot__Fuel Type_Diesel    112723 non-null  float64
 6   onehot__Fuel Type_Electric  112723 non-null  float64
 7   onehot__Fuel Type_Hybrid    112723 non-null  float64
 8   onehot__Fuel Type_Petrol    112723 non-null  float64
 9   onehot__Car Type_Hatchback  112723 non-null  float64
 10  onehot__Car Type_Luxury     112723 non-null  float64
 11  onehot__Car Type_MPV        112723 non-null  float64
 12  onehot__Car Type_SUV        112723 non-null  float64
 13  onehot__Car Ty

In [4]:
y_train.shape

(112723,)

In [5]:
X_test.shape

(28181, 20)

---
## ***2. Applying DBSCAN***

### ***2.1 Base DBSCAN***

In [8]:
# Base model
base_dbscan = DBSCAN()

base_dbscan_labels = base_dbscan.fit_predict(X_train)

In [14]:
count = 0
for label in base_dbscan_labels:
    if label == -1:
        count +=1
print("Total Outliers : ", count)
print("Normal Points : ", len(base_dbscan_labels) - count)
print("Anomalyy Rate : ", np.mean(base_dbscan_labels == -1))

Total Outliers :  34457
Normal Points :  78266
Anomalyy Rate :  0.3056785216859026


### ***2.2 Tuned DBSCAN***

In [21]:
from sklearn.metrics import silhouette_score

best_score = -1
best_params = None

eps_values = [0.3, 0.5, 0.7, 1.0, 1.5]
min_samples_values = [3, 5, 10, 15]

for eps in eps_values:
    for min_samples in min_samples_values:

        dbscan = DBSCAN(
            eps=eps,
            min_samples=min_samples,
            n_jobs=-1  # use all CPU cores
        )

        labels = dbscan.fit_predict(X_train)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        # Skip invalid clustering results
        if n_clusters < 2:
            continue

        score = silhouette_score(
            X_train,
            labels,
            sample_size=10000,
            random_state=42
        )

        outlier_ratio = np.mean(labels == -1)

        print(
            f"eps={eps}, "
            f"min_samples={min_samples}, "
            f"clusters={n_clusters}, "
            f"outliers={outlier_ratio:.2%}, "
            f"silhouette={score:.4f}"
        )

        if score > best_score:
            best_score = score
            best_params = {
                "eps": eps,
                "min_samples": min_samples
            }

print("\nBest Score:", best_score)
print("Best Params:", best_params)

eps=0.3, min_samples=3, clusters=5205, outliers=46.80%, silhouette=-0.3992
eps=0.3, min_samples=5, clusters=2038, outliers=61.78%, silhouette=-0.4472
eps=0.3, min_samples=10, clusters=520, outliers=81.56%, silhouette=-0.4742
eps=0.3, min_samples=15, clusters=208, outliers=90.88%, silhouette=-0.4814
eps=0.5, min_samples=3, clusters=3343, outliers=20.70%, silhouette=-0.2057
eps=0.5, min_samples=5, clusters=1548, outliers=30.57%, silhouette=-0.2468
eps=0.5, min_samples=10, clusters=529, outliers=46.92%, silhouette=-0.3056
eps=0.5, min_samples=15, clusters=321, outliers=56.88%, silhouette=-0.3333
eps=0.7, min_samples=3, clusters=2075, outliers=10.28%, silhouette=-0.0521
eps=0.7, min_samples=5, clusters=1047, outliers=15.76%, silhouette=-0.0716
eps=0.7, min_samples=10, clusters=505, outliers=26.21%, silhouette=-0.1275
eps=0.7, min_samples=15, clusters=311, outliers=34.05%, silhouette=-0.1692
eps=1.0, min_samples=3, clusters=1170, outliers=4.19%, silhouette=0.0918
eps=1.0, min_samples=5, clu

In [26]:
best_dbscan_model = DBSCAN(
    eps=1.5,
    min_samples=3
)

best_dbscan_labels = best_dbscan_model.fit_predict(X_train)

In [27]:
count = 0
for label in best_dbscan_labels:
    if label == -1:
        count +=1
print("Total Outliers : ", count)
print("Normal Points : ", len(best_dbscan_labels) - count)
print("Anomalyy Rate : ", np.mean(best_dbscan_labels == -1))

Total Outliers :  1643
Normal Points :  111080
Anomalyy Rate :  0.014575552460456163


In [25]:
# Save parameters

parameter_df = pd.DataFrame({
    "eps":[1.5],
    "min_samples":3
})
parameter_df

parameter_df.to_csv("../hyperparameters/dbscan_pamas..csv", index=False)

---
## ***2. Applying IsolationForest***

### ***2.1 Base IsolationForest***

In [28]:
base_if = IsolationForest()

base_if_labels = base_if.fit_predict(X_train)

In [29]:
count = 0
for label in base_if_labels:
    if label == -1:
        count +=1
print("Total Outliers : ", count)
print("Normal Points : ", len(base_if_labels) - count)
print("Anomalyy Rate : ", np.mean(base_if_labels == -1))

Total Outliers :  28736
Normal Points :  83987
Anomalyy Rate :  0.25492579154209877


### ***2.2 Tuned IsolationForest***

In [31]:
results = []

for contamination in [0.01, 0.03, 0.05, 0.1]:
    for n_estimators in [100, 200, 300]:
        for max_samples in ['auto', 0.7, 0.8, 1.0]:

            model = IsolationForest(
                contamination=contamination,
                n_estimators=n_estimators,
                max_samples=max_samples,
                random_state=42,
                n_jobs=-1
            )

            model.fit(X_train)

            preds = model.predict(X_train)
            scores = model.decision_function(X_train)

            n_anomalies = np.sum(preds == -1)
            anomaly_ratio = n_anomalies / len(preds)

            score_std = np.std(scores)
            score_range = np.percentile(scores, 95) - np.percentile(scores, 5)

            results.append({
                "contamination": contamination,
                "n_estimators": n_estimators,
                "max_samples": max_samples,
                "n_anomalies": n_anomalies,
                "anomaly_ratio": anomaly_ratio,
                "score_std": score_std,
                "score_range_5_95": score_range
            })

            print(
                f"cont={contamination:<4} | "
                f"trees={n_estimators:<3} | "
                f"max_samples={str(max_samples):<4} | "
                f"anomalies={n_anomalies:<6} | "
                f"ratio={anomaly_ratio:.2%} | "
                f"std={score_std:.4f} | "
                f"range={score_range:.4f}"
            )

results_df = pd.DataFrame(results)

print("\nTop 10 by score range:")
print(
    results_df.sort_values(
        by="score_range_5_95",
        ascending=False
    ).head(10)
)

cont=0.01 | trees=100 | max_samples=auto | anomalies=1128   | ratio=1.00% | std=0.0469 | range=0.1543
cont=0.01 | trees=100 | max_samples=0.7  | anomalies=1128   | ratio=1.00% | std=0.0454 | range=0.1462
cont=0.01 | trees=100 | max_samples=0.8  | anomalies=1128   | ratio=1.00% | std=0.0451 | range=0.1455
cont=0.01 | trees=100 | max_samples=1.0  | anomalies=1128   | ratio=1.00% | std=0.0438 | range=0.1414
cont=0.01 | trees=200 | max_samples=auto | anomalies=1128   | ratio=1.00% | std=0.0453 | range=0.1474
cont=0.01 | trees=200 | max_samples=0.7  | anomalies=1128   | ratio=1.00% | std=0.0453 | range=0.1462
cont=0.01 | trees=200 | max_samples=0.8  | anomalies=1128   | ratio=1.00% | std=0.0450 | range=0.1459
cont=0.01 | trees=200 | max_samples=1.0  | anomalies=1128   | ratio=1.00% | std=0.0434 | range=0.1392
cont=0.01 | trees=300 | max_samples=auto | anomalies=1128   | ratio=1.00% | std=0.0456 | range=0.1479
cont=0.01 | trees=300 | max_samples=0.7  | anomalies=1128   | ratio=1.00% | std=0.

In [ ]:
best_if_model = IsolationForest(
    eps=1.5,
    min_samples=3
)

best_dbscan_labels = best_dbscan_model.fit_predict(X_train)

In [36]:
# Save parameters

parameter_df = pd.DataFrame({
    "contamination":[0.03],
    "n_estimators":[100],
    "max_samples":["auto"]
})

parameter_df.to_csv("../hyperparameters/isolationforest_pamas..csv", index=False)
parameter_df

,contamination,n_estimators,max_samples
0,0.03,100,auto


---
## ***2. Applying LOF***

### ***2.1 Base LOF***

In [ ]:
base_lof = LocalOutlierFactor()

base_lof_labels = base_lof.fit_predit(X_train)

In [ ]:
count = 0
for label in base_lof_labels:
    if label == -1:
        count +=1
print("Total Outliers : ", count)
print("Normal Points : ", len(base_lof_labels) - count)
print("Anomalyy Rate : ", np.mean(base_lof_labels == -1))

### ***2.2 Tuned LOF***

In [37]:
results = []

for n_neighbors in [5, 10, 20, 30, 50]:
    for contamination in [0.01, 0.03, 0.05, 0.1]:

        lof = LocalOutlierFactor(
            n_neighbors=n_neighbors,
            contamination=contamination,
            n_jobs=-1
        )

        preds = lof.fit_predict(X_train)

        scores = lof.negative_outlier_factor_

        n_anomalies = np.sum(preds == -1)
        anomaly_ratio = n_anomalies / len(preds)

        score_std = np.std(scores)
        score_range = (
            np.percentile(scores, 95)
            - np.percentile(scores, 5)
        )

        results.append({
            "n_neighbors": n_neighbors,
            "contamination": contamination,
            "n_anomalies": n_anomalies,
            "anomaly_ratio": anomaly_ratio,
            "score_std": score_std,
            "score_range_5_95": score_range
        })

        print(
            f"neighbors={n_neighbors:<2} | "
            f"cont={contamination:<4} | "
            f"anomalies={n_anomalies:<6} | "
            f"ratio={anomaly_ratio:.2%} | "
            f"std={score_std:.4f} | "
            f"range={score_range:.4f}"
        )

results_df = pd.DataFrame(results)

print("\nTop 10 by score range:")
print(
    results_df.sort_values(
        by="score_range_5_95",
        ascending=False
    ).head(10)
)

neighbors=5  | cont=0.01 | anomalies=1128   | ratio=1.00% | std=0.1504 | range=0.3575
neighbors=5  | cont=0.03 | anomalies=3382   | ratio=3.00% | std=0.1504 | range=0.3575
neighbors=5  | cont=0.05 | anomalies=5637   | ratio=5.00% | std=0.1504 | range=0.3575
neighbors=5  | cont=0.1  | anomalies=11273  | ratio=10.00% | std=0.1504 | range=0.3575
neighbors=10 | cont=0.01 | anomalies=1128   | ratio=1.00% | std=0.1225 | range=0.2758
neighbors=10 | cont=0.03 | anomalies=3382   | ratio=3.00% | std=0.1225 | range=0.2758
neighbors=10 | cont=0.05 | anomalies=5637   | ratio=5.00% | std=0.1225 | range=0.2758
neighbors=10 | cont=0.1  | anomalies=11273  | ratio=10.00% | std=0.1225 | range=0.2758
neighbors=20 | cont=0.01 | anomalies=1128   | ratio=1.00% | std=0.1146 | range=0.2551
neighbors=20 | cont=0.03 | anomalies=3382   | ratio=3.00% | std=0.1146 | range=0.2551
neighbors=20 | cont=0.05 | anomalies=5637   | ratio=5.00% | std=0.1146 | range=0.2551
neighbors=20 | cont=0.1  | anomalies=11273  | ratio=

In [41]:
# Svaing params

param_df = pd.DataFrame({
    "n_neighbors":[5],
    "contamination":[0.03]
})

param_df.to_csv("../hyperparameters/lof_parameters.csv", index=False)
param_df

,n_neighbors,contamination
0,5,0.03
